# Exploratory Data Analysis: MCO vs MIA

Interactive visualizations comparing Orlando (MCO) and Miami (MIA) airports (2004-2008).

**Team E:** EL MOAZEN RAMI - THEOFANOPOULOS MICHAIL - KITSAKIS GEORGIOS - SCHOINAS VASILEIOS

## Setup

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

print(f"pandas {pd.__version__}")
print(f"plotly {plotly.__version__}")

pandas 2.2.2
plotly 5.24.1


## Load Cleaned Data

In [ ]:
# Load cleaned data from notebook 02
df = pd.read_csv('../data/processed/mco_mia_clean.csv.gz', low_memory=False)

# Ensure FlightDate is datetime
df['FlightDate'] = pd.to_datetime(df['FlightDate'])

print(f"Loaded {len(df):,} flights")
print(f"Date range: {df['FlightDate'].min().date()} to {df['FlightDate'].max().date()}")
print(f"Airports: MCO, MIA")

df.head()

Loaded 1,592,198 flights
Date range: 2004-01-01 to 2008-04-30
Airports: MCO, MIA


,Year,Month,DayofMonth,DayOfWeek,DepTime,CRSDepTime,ArrTime,CRSArrTime,UniqueCarrier,FlightNum,TailNum,ActualElapsedTime,CRSElapsedTime,AirTime,ArrDelay,DepDelay,Origin,Dest,Distance,TaxiIn,TaxiOut,Cancelled,CancellationCode,Diverted,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay,FlightDate,CRSDepTime_Hour,CRSDepTime_Min,DepTime_Hour,DepTime_Min,CRSArrTime_Hour,CRSArrTime_Min,ArrTime_Hour,ArrTime_Min,OnTime,DelayCategory,Quarter,IsHolidaySeason,DayName,IsMCO,IsMIA,MCO_Direction,MIA_Direction,TimeOfDay,IsWeekend,DistanceCategory,Season,PrimaryDelayCause
0,2004,1,1,4,1622.0,1625,1955.0,2003,UA,470,N501UA,153.0,158.0,128.0,-8.0,-3.0,ORD,MCO,1005,2.0,23.0,0,NaN,0,0.0,0.0,0.0,0.0,0.0,2004-01-01,16,25,16.0,22.0,20,3,19.0,55.0,True,Early,1,False,Thu,True,False,Arrival,NaN,Afternoon,False,Long,Winter,NaN
1,2004,1,2,5,1641.0,1625,2033.0,2003,UA,470,N514UA,172.0,158.0,123.0,30.0,16.0,ORD,MCO,1005,2.0,47.0,0,NaN,0,0.0,0.0,14.0,0.0,16.0,2004-01-02,16,25,16.0,41.0,20,3,20.0,33.0,False,Delayed,1,False,Fri,True,False,Arrival,NaN,Afternoon,False,Long,Winter,LateAircraftDelay
2,2004,1,3,6,1644.0,1625,2000.0,2003,UA,470,N504UA,136.0,158.0,120.0,-3.0,19.0,ORD,MCO,1005,3.0,13.0,0,NaN,0,0.0,0.0,0.0,0.0,0.0,2004-01-03,16,25,16.0,44.0,20,3,20.0,0.0,True,Early,1,False,Sat,True,False,Arrival,NaN,Afternoon,True,Long,Winter,NaN
3,2004,1,4,7,1744.0,1625,2144.0,2003,UA,470,N558UA,180.0,158.0,128.0,101.0,79.0,ORD,MCO,1005,4.0,48.0,0,NaN,0,12.0,0.0,22.0,0.0,67.0,2004-01-04,16,25,17.0,44.0,20,3,21.0,44.0,False,SeverelyDelayed,1,False,Sun,True,False,Arrival,NaN,Afternoon,True,Long,Winter,LateAircraftDelay
4,2004,1,5,1,1637.0,1625,2006.0,2003,UA,470,N525UA,149.0,158.0,131.0,3.0,12.0,ORD,MCO,1005,4.0,14.0,0,NaN,0,0.0,0.0,0.0,0.0,0.0,2004-01-05,16,25,16.0,37.0,20,3,20.0,6.0,True,OnTime,1,False,Mon,True,False,Arrival,NaN,Afternoon,False,Long,Winter,CarrierDelay


## Color Scheme

Consistent colors for better storytelling:

In [ ]:
# Color palette
MCO_COLOR = '#3498db'  # Blue - Orlando
MIA_COLOR = '#e74c3c'  # Red - Miami
NEUTRAL_COLOR = '#95a5a6'  # Gray

# Template
TEMPLATE = 'plotly_white'

# Airport names dictionary (major US airports)
AIRPORT_NAMES = {
    'ATL': 'Atlanta', 'ORD': 'Chicago O\'Hare', 'DFW': 'Dallas/Fort Worth',
    'LAX': 'Los Angeles', 'DEN': 'Denver', 'JFK': 'New York JFK',
    'SFO': 'San Francisco', 'LAS': 'Las Vegas', 'SEA': 'Seattle',
    'PHX': 'Phoenix', 'IAH': 'Houston', 'BOS': 'Boston',
    'EWR': 'Newark', 'MSP': 'Minneapolis', 'DTW': 'Detroit',
    'PHL': 'Philadelphia', 'LGA': 'New York LaGuardia', 'FLL': 'Fort Lauderdale',
    'BWI': 'Baltimore', 'DCA': 'Washington Reagan', 'IAD': 'Washington Dulles',
    'SLC': 'Salt Lake City', 'MDW': 'Chicago Midway', 'SAN': 'San Diego',
    'TPA': 'Tampa', 'PDX': 'Portland', 'STL': 'St. Louis',
    'HNL': 'Honolulu', 'CLT': 'Charlotte', 'PIT': 'Pittsburgh',
    'MCO': 'Orlando', 'MIA': 'Miami', 'RDU': 'Raleigh-Durham',
    'AUS': 'Austin', 'MCI': 'Kansas City', 'CVG': 'Cincinnati',
    'CLE': 'Cleveland', 'IND': 'Indianapolis', 'CMH': 'Columbus',
    'MKE': 'Milwaukee', 'OAK': 'Oakland', 'SJC': 'San Jose',
    'SMF': 'Sacramento', 'SNA': 'Orange County', 'BNA': 'Nashville',
    'MSY': 'New Orleans', 'RSW': 'Fort Myers', 'PBI': 'West Palm Beach',
    'BDL': 'Hartford', 'BUF': 'Buffalo', 'ONT': 'Ontario CA',
    'ABQ': 'Albuquerque', 'JAX': 'Jacksonville', 'PVD': 'Providence',
    'SJU': 'San Juan PR', 'RNO': 'Reno', 'TUL': 'Tulsa'
}

# Airline names dictionary
AIRLINE_NAMES = {
    'AA': 'American Airlines', 'DL': 'Delta Air Lines', 'UA': 'United Airlines',
    'WN': 'Southwest Airlines', 'US': 'US Airways', 'CO': 'Continental Airlines',
    'NW': 'Northwest Airlines', 'B6': 'JetBlue Airways', 'AS': 'Alaska Airlines',
    'F9': 'Frontier Airlines', 'FL': 'AirTran Airways', 'HP': 'America West',
    'TZ': 'ATA Airlines', 'OH': 'Comair', 'MQ': 'American Eagle',
    'OO': 'SkyWest Airlines', 'EV': 'ExpressJet', 'XE': 'ExpressJet',
    'YV': 'Mesa Airlines', '9E': 'Pinnacle Airlines', 'HA': 'Hawaiian Airlines'
}

print("Colors set: MCO (Blue), MIA (Red)")

Colors set: MCO (Blue), MIA (Red)


---
# 1. Temporal Analysis

## 1.1 Monthly Flight Volume (MCO vs MIA)

In [ ]:
# Group by year-month and airport
df['YearMonth'] = df['FlightDate'].dt.to_period('M').astype(str)

# Departures only
departures = df[df['Origin'].isin(['MCO', 'MIA'])]

monthly_volume = departures.groupby(['YearMonth', 'Origin']).size().reset_index(name='Flights')

# Create figure
fig = go.Figure()

# MCO line
mco_data = monthly_volume[monthly_volume['Origin'] == 'MCO']
fig.add_trace(go.Scatter(
    x=mco_data['YearMonth'],
    y=mco_data['Flights'],
    mode='lines+markers',
    name='MCO (Orlando)',
    line=dict(color=MCO_COLOR, width=3),
    marker=dict(size=6),
    hovertemplate='<b>MCO</b><br>%{x}<br>Flights: %{y:,}<extra></extra>'
))

# MIA line
mia_data = monthly_volume[monthly_volume['Origin'] == 'MIA']
fig.add_trace(go.Scatter(
    x=mia_data['YearMonth'],
    y=mia_data['Flights'],
    mode='lines+markers',
    name='MIA (Miami)',
    line=dict(color=MIA_COLOR, width=3),
    marker=dict(size=6),
    hovertemplate='<b>MIA</b><br>%{x}<br>Flights: %{y:,}<extra></extra>'
))

fig.update_layout(
    title='Monthly Departures: MCO vs MIA (2004-2008)',
    xaxis_title='Month',
    yaxis_title='Number of Departures',
    template=TEMPLATE,
    height=500,
    hovermode='x unified',
    legend=dict(x=0.02, y=0.98, bgcolor='rgba(255,255,255,0.8)')
)

fig.show()

### 📊 Key Insights:
- **MCO consistently has ~2x more departures** than MIA throughout the period
- **Seasonal patterns visible**: Summer months (Jun-Aug) show increased traffic at both airports
- **2008 data is incomplete** (Jan-Apr only), showing the expected drop-off
- Both airports show **steady growth from 2004-2007**

## 1.2 Quarterly Trends Over Years

**What this shows:** Flight volume grouped by quarters (Q1=Jan-Mar, Q2=Apr-Jun, Q3=Jul-Sep, Q4=Oct-Dec) to see seasonal patterns and growth over time.

In [ ]:
# Group by year, quarter, and airport
quarterly = departures.groupby(['Year', 'Quarter', 'Origin']).size().reset_index(name='Flights')
quarterly['Period'] = quarterly['Year'].astype(str) + ' Q' + quarterly['Quarter'].astype(str)

# Create grouped bar chart
fig = px.bar(
    quarterly,
    x='Period',
    y='Flights',
    color='Origin',
    barmode='group',
    color_discrete_map={'MCO': MCO_COLOR, 'MIA': MIA_COLOR},
    labels={'Flights': 'Number of Departures', 'Period': 'Quarter'},
    title='Quarterly Flight Volume: MCO vs MIA<br><sub>Q1=Jan-Mar, Q2=Apr-Jun, Q3=Jul-Sep, Q4=Oct-Dec</sub>',
    template=TEMPLATE,
    height=500
)

fig.update_layout(
    xaxis_tickangle=-45,
    legend_title_text='Airport',
    hovermode='x unified'
)

fig.show()

### 📊 Key Insights:
- **Q3 (Summer) is the busiest quarter** for both airports - peak tourism season
- **Q1 (Winter) is also strong** - likely snowbirds and winter vacations to Florida
- **Q2 and Q4 show lower volumes** - off-peak travel seasons
- MCO maintains its 2x volume advantage **consistently across all quarters**

## 1.3 Day of Week Patterns

In [ ]:
# Average daily flights by day of week
day_order = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

dow_pattern = departures.groupby(['DayName', 'Origin']).size().reset_index(name='TotalFlights')

# Calculate average per day (total flights / number of occurrences)
# Count unique dates for each day of week
date_counts = departures.groupby(['DayName', 'Origin'])['FlightDate'].nunique().reset_index(name='NumDays')
dow_pattern = dow_pattern.merge(date_counts, on=['DayName', 'Origin'])
dow_pattern['AvgFlights'] = (dow_pattern['TotalFlights'] / dow_pattern['NumDays']).round(0)

# Ensure proper order
dow_pattern['DayName'] = pd.Categorical(dow_pattern['DayName'], categories=day_order, ordered=True)
dow_pattern = dow_pattern.sort_values(['Origin', 'DayName'])

# Create figure
fig = px.bar(
    dow_pattern,
    x='DayName',
    y='AvgFlights',
    color='Origin',
    barmode='group',
    color_discrete_map={'MCO': MCO_COLOR, 'MIA': MIA_COLOR},
    labels={'AvgFlights': 'Average Daily Departures', 'DayName': 'Day of Week'},
    title='Average Departures by Day of Week',
    template=TEMPLATE,
    height=500
)

fig.update_layout(
    legend_title_text='Airport'
)

fig.show()

### 📊 Key Insights:
- **Weekend traffic is lower** than weekdays at both airports
- **Friday is the busiest day** - likely business travelers returning home
- **Sunday shows increased traffic** compared to Saturday - end of weekend travel
- Pattern suggests **strong business travel component** alongside leisure

---
# 2. Airport Comparison

## 2.1 Top 10 Destinations from MCO vs MIA

In [ ]:
# Get top 10 destinations for each airport
mco_top = departures[departures['Origin'] == 'MCO']['Dest'].value_counts().head(10).reset_index()
mco_top.columns = ['Destination', 'Flights']
mco_top['Airport'] = 'MCO'
mco_top['DestName'] = mco_top['Destination'].map(AIRPORT_NAMES).fillna(mco_top['Destination'])

mia_top = departures[departures['Origin'] == 'MIA']['Dest'].value_counts().head(10).reset_index()
mia_top.columns = ['Destination', 'Flights']
mia_top['Airport'] = 'MIA'
mia_top['DestName'] = mia_top['Destination'].map(AIRPORT_NAMES).fillna(mia_top['Destination'])

# Combine
top_dest = pd.concat([mco_top, mia_top])

# Create subplots
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Top 10 Destinations from MCO', 'Top 10 Destinations from MIA'),
    horizontal_spacing=0.15
)

# MCO bars
fig.add_trace(
    go.Bar(
        x=mco_top['Flights'],
        y=mco_top['Destination'],
        orientation='h',
        marker_color=MCO_COLOR,
        name='MCO',
        text=mco_top['Flights'],
        texttemplate='%{text:,}',
        textposition='outside',
        hovertemplate='<b>%{y} - ' + mco_top['DestName'] + '</b><br>Flights: %{x:,}<extra></extra>',
        customdata=mco_top['DestName']
    ),
    row=1, col=1
)

# MIA bars
fig.add_trace(
    go.Bar(
        x=mia_top['Flights'],
        y=mia_top['Destination'],
        orientation='h',
        marker_color=MIA_COLOR,
        name='MIA',
        text=mia_top['Flights'],
        texttemplate='%{text:,}',
        textposition='outside',
        hovertemplate='<b>%{y} - ' + mia_top['DestName'] + '</b><br>Flights: %{x:,}<extra></extra>',
        customdata=mia_top['DestName']
    ),
    row=1, col=2
)

fig.update_xaxes(title_text='Number of Flights', row=1, col=1)
fig.update_xaxes(title_text='Number of Flights', row=1, col=2)
fig.update_yaxes(title_text='Destination', autorange='reversed', row=1, col=1)
fig.update_yaxes(title_text='Destination', autorange='reversed', row=1, col=2)

fig.update_layout(
    title_text='Top Destinations Comparison (2004-2008)',
    showlegend=False,
    template=TEMPLATE,
    height=600
)

fig.show()

### 📊 Key Insights:
- **MCO focuses on domestic hubs**: ATL, ORD, DFW dominate - connecting passengers
- **MIA shows different pattern**: NY gateways (JFK, LGA, EWR) are top destinations
- **ATL is #1 for both** but with different volumes (71K vs 30K)
- MCO serves more **diverse route network** (broader geographic spread)

## 2.2 Airline Market Share

In [ ]:
# Top 8 airlines for each airport
mco_airlines = departures[departures['Origin'] == 'MCO']['UniqueCarrier'].value_counts().head(8).reset_index()
mco_airlines.columns = ['Airline', 'Flights']
mco_airlines['AirlineName'] = mco_airlines['Airline'].map(AIRLINE_NAMES).fillna(mco_airlines['Airline'])
mco_airlines['LegendLabel'] = mco_airlines['Airline'] + ' - ' + mco_airlines['AirlineName']

mia_airlines = departures[departures['Origin'] == 'MIA']['UniqueCarrier'].value_counts().head(8).reset_index()
mia_airlines.columns = ['Airline', 'Flights']
mia_airlines['AirlineName'] = mia_airlines['Airline'].map(AIRLINE_NAMES).fillna(mia_airlines['Airline'])
mia_airlines['LegendLabel'] = mia_airlines['Airline'] + ' - ' + mia_airlines['AirlineName']

# Create figure with Plotly Express for automatic legend
from plotly import graph_objects as go

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type':'pie'}, {'type':'pie'}]],
    subplot_titles=('MCO (Orlando) - Top Airlines', 'MIA (Miami) - Top Airlines')
)

# MCO pie
fig.add_trace(
    go.Pie(
        labels=mco_airlines['Airline'],  # Short labels on pie
        values=mco_airlines['Flights'],
        name='MCO',
        marker_colors=px.colors.sequential.Blues_r,
        textinfo='label+percent',
        hovertemplate='<b>%{label} - ' + mco_airlines['AirlineName'] + '</b><br>Flights: %{value:,}<br>Share: %{percent}<extra></extra>'
    ),
    row=1, col=1
)

# MIA pie
fig.add_trace(
    go.Pie(
        labels=mia_airlines['Airline'],  # Short labels on pie
        values=mia_airlines['Flights'],
        name='MIA',
        marker_colors=px.colors.sequential.Reds_r,
        textinfo='label+percent',
        hovertemplate='<b>%{label} - ' + mia_airlines['AirlineName'] + '</b><br>Flights: %{value:,}<br>Share: %{percent}<extra></extra>'
    ),
    row=1, col=2
)

fig.update_layout(
    title_text='Airline Market Share (2004-2008)',
    template=TEMPLATE,
    height=500
)

fig.show()

### 📊 Key Insights:
- **MCO has balanced competition**: WN (31%), DL (14%), FL (14%) - no single dominant carrier
- **MIA is AA-dominated (64%)**: American Airlines has massive market concentration
- **Legacy carriers strong at both**: Delta, United, Continental all present
- **Low-cost carriers (WN, FL, B6)** have stronger presence at MCO

## 2.3 Departure Time Heatmap

In [ ]:
# Group by hour and day of week
departure_heatmap = departures.groupby(['Origin', 'DayName', 'CRSDepTime_Hour']).size().reset_index(name='Flights')

# Ensure proper day order
departure_heatmap['DayName'] = pd.Categorical(departure_heatmap['DayName'], categories=day_order, ordered=True)
departure_heatmap = departure_heatmap.sort_values('DayName')

# Create pivot tables
mco_pivot = departure_heatmap[departure_heatmap['Origin'] == 'MCO'].pivot(index='DayName', columns='CRSDepTime_Hour', values='Flights').fillna(0)
mia_pivot = departure_heatmap[departure_heatmap['Origin'] == 'MIA'].pivot(index='DayName', columns='CRSDepTime_Hour', values='Flights').fillna(0)

# Create subplots
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('MCO (Orlando) - Departure Times', 'MIA (Miami) - Departure Times'),
    vertical_spacing=0.12
)

# MCO heatmap
fig.add_trace(
    go.Heatmap(
        z=mco_pivot.values,
        x=mco_pivot.columns,
        y=mco_pivot.index,
        colorscale='Blues',
        name='MCO',
        hovertemplate='Day: %{y}<br>Hour: %{x}:00<br>Flights: %{z:,.0f}<extra></extra>',
        showscale=True,
        colorbar=dict(x=1.12, len=0.45, y=0.75)
    ),
    row=1, col=1
)

# MIA heatmap
fig.add_trace(
    go.Heatmap(
        z=mia_pivot.values,
        x=mia_pivot.columns,
        y=mia_pivot.index,
        colorscale='Reds',
        name='MIA',
        hovertemplate='Day: %{y}<br>Hour: %{x}:00<br>Flights: %{z:,.0f}<extra></extra>',
        showscale=True,
        colorbar=dict(x=1.12, len=0.45, y=0.25)
    ),
    row=2, col=1
)

fig.update_xaxes(title_text='Hour of Day', row=1, col=1)
fig.update_xaxes(title_text='Hour of Day', row=2, col=1)
fig.update_yaxes(title_text='Day of Week', row=1, col=1)
fig.update_yaxes(title_text='Day of Week', row=2, col=1)

fig.update_layout(
    title_text='Flight Departure Patterns by Day & Hour',
    template=TEMPLATE,
    height=700
)

fig.show()

### 📊 Key Insights:
- **Peak hours: 6-8 AM and 5-7 PM** for both airports (business travel times)
- **MCO has more evening flights** - accommodating theme park visitors
- **Weekends show different patterns**: More evenly distributed throughout the day
- **Early morning (5-6 AM) is busiest** - airlines maximizing aircraft utilization

---
# 3. Performance Metrics

## 3.1 Average Delay by Month (MCO vs MIA)

In [ ]:
# Calculate average delay by month (exclude cancelled)
non_cancelled = departures[departures['Cancelled'] == 0].copy()
monthly_delay = non_cancelled.groupby(['YearMonth', 'Origin'])['ArrDelay'].mean().reset_index()

# Create figure
fig = go.Figure()

# MCO line
mco_delay = monthly_delay[monthly_delay['Origin'] == 'MCO']
fig.add_trace(go.Scatter(
    x=mco_delay['YearMonth'],
    y=mco_delay['ArrDelay'],
    mode='lines+markers',
    name='MCO (Orlando)',
    line=dict(color=MCO_COLOR, width=3),
    marker=dict(size=6),
    hovertemplate='<b>MCO</b><br>%{x}<br>Avg Delay: %{y:.1f} min<extra></extra>'
))

# MIA line
mia_delay = monthly_delay[monthly_delay['Origin'] == 'MIA']
fig.add_trace(go.Scatter(
    x=mia_delay['YearMonth'],
    y=mia_delay['ArrDelay'],
    mode='lines+markers',
    name='MIA (Miami)',
    line=dict(color=MIA_COLOR, width=3),
    marker=dict(size=6),
    hovertemplate='<b>MIA</b><br>%{x}<br>Avg Delay: %{y:.1f} min<extra></extra>'
))

# Add zero line
fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)

fig.update_layout(
    title='Average Arrival Delay by Month (2004-2008)',
    xaxis_title='Month',
    yaxis_title='Average Delay (minutes)',
    template=TEMPLATE,
    height=500,
    hovermode='x unified',
    legend=dict(x=0.02, y=0.98, bgcolor='rgba(255,255,255,0.8)')
)

fig.show()

### 📊 Key Insights:
- **MIA has consistently higher delays** than MCO (avg 11 vs 8 minutes)
- **Summer 2007 shows major spike** - likely weather-related (hurricane season)
- **Winter months generally better** for on-time performance
- **2008 shows improvement** at both airports (though partial data)

## 3.2 On-Time Performance Trend

In [ ]:
# Calculate OTP by month
monthly_otp = non_cancelled.groupby(['YearMonth', 'Origin'])['OnTime'].mean().reset_index()
monthly_otp['OTP_Percent'] = monthly_otp['OnTime'] * 100

# Create figure
fig = go.Figure()

# MCO line
mco_otp = monthly_otp[monthly_otp['Origin'] == 'MCO']
fig.add_trace(go.Scatter(
    x=mco_otp['YearMonth'],
    y=mco_otp['OTP_Percent'],
    mode='lines+markers',
    name='MCO (Orlando)',
    line=dict(color=MCO_COLOR, width=3),
    marker=dict(size=6),
    fill='tozeroy',
    fillcolor=f'rgba(52, 152, 219, 0.1)',
    hovertemplate='<b>MCO</b><br>%{x}<br>OTP: %{y:.1f}%<extra></extra>'
))

# MIA line
mia_otp = monthly_otp[monthly_otp['Origin'] == 'MIA']
fig.add_trace(go.Scatter(
    x=mia_otp['YearMonth'],
    y=mia_otp['OTP_Percent'],
    mode='lines+markers',
    name='MIA (Miami)',
    line=dict(color=MIA_COLOR, width=3),
    marker=dict(size=6),
    fill='tozeroy',
    fillcolor=f'rgba(231, 76, 60, 0.1)',
    hovertemplate='<b>MIA</b><br>%{x}<br>OTP: %{y:.1f}%<extra></extra>'
))

# Add 80% benchmark line
fig.add_hline(y=80, line_dash="dash", line_color="green", opacity=0.5,
              annotation_text="80% Target", annotation_position="right")

fig.update_layout(
    title='On-Time Performance Trend (≤15 min delay)',
    xaxis_title='Month',
    yaxis_title='On-Time Performance (%)',
    template=TEMPLATE,
    height=500,
    hovermode='x unified',
    legend=dict(x=0.02, y=0.02, bgcolor='rgba(255,255,255,0.8)')
)

fig.show()

### 📊 Key Insights:
- **MCO exceeds 80% OTP target** most months (~79% average)
- **MIA falls below target** (~74% average) - consistent performance gap
- **Both airports show declining OTP 2004-2007** - system-wide congestion
- **Seasonal dips in summer** correlate with weather and high traffic

## 3.3 Delay Causes Breakdown

In [ ]:
# Get delayed flights only (ArrDelay > 15)
delayed_flights = non_cancelled[non_cancelled['ArrDelay'] > 15].copy()

# Sum delay minutes by cause
delay_cols = ['CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay']

mco_delays = delayed_flights[delayed_flights['Origin'] == 'MCO'][delay_cols].sum()
mia_delays = delayed_flights[delayed_flights['Origin'] == 'MIA'][delay_cols].sum()

# Clean names - short for pie chart
cause_names = ['Carrier', 'Weather', 'Airport Control', 'Security', 'Late Aircraft']
mco_delays.index = cause_names
mia_delays.index = cause_names

# Create subplots
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type':'pie'}, {'type':'pie'}]],
    subplot_titles=('MCO Delay Causes', 'MIA Delay Causes')
)

# Color scheme for causes
colors = ['#3498db', '#e74c3c', '#f39c12', '#9b59b6', '#1abc9c']

# MCO pie
fig.add_trace(
    go.Pie(
        labels=mco_delays.index,
        values=mco_delays.values,
        name='MCO',
        marker_colors=colors,
        textinfo='label+percent',
        hovertemplate='<b>%{label}</b><br>Total Minutes: %{value:,.0f}<br>Share: %{percent}<extra></extra>'
    ),
    row=1, col=1
)

# MIA pie
fig.add_trace(
    go.Pie(
        labels=mia_delays.index,
        values=mia_delays.values,
        name='MIA',
        marker_colors=colors,
        textinfo='label+percent',
        hovertemplate='<b>%{label}</b><br>Total Minutes: %{value:,.0f}<br>Share: %{percent}<extra></extra>'
    ),
    row=1, col=2
)

fig.update_layout(
    title_text='Primary Causes of Delays (2004-2008)<br><sub>NAS = National Aviation System (Air Traffic Control, Airport Operations)</sub>',
    template=TEMPLATE,
    height=500
)

fig.show()

### 📊 Key Insights:
- **Late Aircraft is #1 cause** at both airports (~40%) - cascading delays
- **NAS (Air Traffic Control) is #2** (~34% MCO, ~35% MIA) - system capacity issues
- **Carrier issues ~21-35%** - airline operational problems
- **Weather and Security are minimal** (<5% combined) - surprisingly low

---
# 4. Operational Insights

## 4.1 Flight Distance Distribution

In [ ]:
# Create overlapping histograms
fig = go.Figure()

# MCO distances
mco_dist = departures[departures['Origin'] == 'MCO']['Distance']
fig.add_trace(go.Histogram(
    x=mco_dist,
    name='MCO (Orlando)',
    marker_color=MCO_COLOR,
    opacity=0.7,
    nbinsx=50,
    hovertemplate='Distance: %{x} mi<br>Flights: %{y:,}<extra></extra>'
))

# MIA distances
mia_dist = departures[departures['Origin'] == 'MIA']['Distance']
fig.add_trace(go.Histogram(
    x=mia_dist,
    name='MIA (Miami)',
    marker_color=MIA_COLOR,
    opacity=0.7,
    nbinsx=50,
    hovertemplate='Distance: %{x} mi<br>Flights: %{y:,}<extra></extra>'
))

# Overlay both histograms
fig.update_layout(
    barmode='overlay',
    title='Flight Distance Distribution',
    xaxis_title='Distance (miles)',
    yaxis_title='Number of Flights',
    template=TEMPLATE,
    height=500,
    legend=dict(x=0.7, y=0.98, bgcolor='rgba(255,255,255,0.8)')
)

fig.show()

# Print summary stats
print(f"MCO - Avg Distance: {mco_dist.mean():.0f} miles (Median: {mco_dist.median():.0f})")
print(f"MIA - Avg Distance: {mia_dist.mean():.0f} miles (Median: {mia_dist.median():.0f})")

MCO - Avg Distance: 902 miles (Median: 895)
MIA - Avg Distance: 1037 miles (Median: 1021)


### 📊 Key Insights:
- **MIA has longer average flights** (1037 miles vs 902 miles)
- **Both airports show peak at ~1000 miles** - typical US domestic range
- **MCO has more short-haul flights** - serving nearby Southeast destinations
- **MIA serves more transcontinental routes** - gateway to South/Central America

## 4.2 Cancellation Rate by Season

In [ ]:
# Calculate cancellation rate by season
season_order = ['Winter', 'Spring', 'Summer', 'Fall']

seasonal_cancel = departures.groupby(['Season', 'Origin']).agg(
    TotalFlights=('Cancelled', 'count'),
    Cancelled=('Cancelled', 'sum')
).reset_index()

seasonal_cancel['CancellationRate'] = (seasonal_cancel['Cancelled'] / seasonal_cancel['TotalFlights']) * 100

# Ensure season order
seasonal_cancel['Season'] = pd.Categorical(seasonal_cancel['Season'], categories=season_order, ordered=True)
seasonal_cancel = seasonal_cancel.sort_values('Season')

# Create grouped bar chart
fig = px.bar(
    seasonal_cancel,
    x='Season',
    y='CancellationRate',
    color='Origin',
    barmode='group',
    color_discrete_map={'MCO': MCO_COLOR, 'MIA': MIA_COLOR},
    labels={'CancellationRate': 'Cancellation Rate (%)', 'Season': 'Season'},
    title='Flight Cancellation Rate by Season',
    template=TEMPLATE,
    height=500,
    text='CancellationRate'
)

fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')

fig.update_layout(
    legend_title_text='Airport'
)

fig.show()

### 📊 Key Insights:
- **Winter has highest cancellations** (~1.7-2.2%) - weather impact
- **MIA cancellation rate higher** across all seasons
- **Summer is most reliable** (<1% cancellations) despite hurricane season
- **Spring and Fall are optimal** for travel reliability

## 4.3 Peak Hours Analysis

In [ ]:
# Average flights per hour
hourly_pattern = departures.groupby(['CRSDepTime_Hour', 'Origin']).size().reset_index(name='TotalFlights')

# Calculate average (divide by number of unique days)
num_days = departures.groupby('Origin')['FlightDate'].nunique().to_dict()
hourly_pattern['AvgFlights'] = hourly_pattern.apply(
    lambda row: row['TotalFlights'] / num_days[row['Origin']], axis=1
)

# Create line chart
fig = px.line(
    hourly_pattern,
    x='CRSDepTime_Hour',
    y='AvgFlights',
    color='Origin',
    color_discrete_map={'MCO': MCO_COLOR, 'MIA': MIA_COLOR},
    markers=True,
    labels={'AvgFlights': 'Average Departures per Day', 'CRSDepTime_Hour': 'Hour of Day'},
    title='Hourly Departure Pattern (Average Daily)',
    template=TEMPLATE,
    height=500
)

fig.update_traces(line=dict(width=3), marker=dict(size=8))

fig.update_layout(
    legend_title_text='Airport',
    xaxis=dict(tickmode='linear', tick0=0, dtick=2)
)

fig.show()

### 📊 Key Insights:
- **Morning rush: 5-7 AM** - highest departure volume
- **Evening peak: 5-8 PM** - second wave of departures
- **Midday lull: 11 AM - 3 PM** - lowest activity
- **MCO has higher peaks** - more concentrated scheduling

## 4.4 Weekend vs Weekday Performance

In [ ]:
# Compare weekend vs weekday metrics
weekend_comparison = non_cancelled.groupby(['Origin', 'IsWeekend']).agg(
    AvgDelay=('ArrDelay', 'mean'),
    OTP=('OnTime', lambda x: x.mean() * 100),
    TotalFlights=('FlightDate', 'count')
).reset_index()

weekend_comparison['DayType'] = weekend_comparison['IsWeekend'].map({True: 'Weekend', False: 'Weekday'})

# Create subplots
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Average Delay', 'On-Time Performance'),
    horizontal_spacing=0.15
)

# Average Delay
for airport, color in [('MCO', MCO_COLOR), ('MIA', MIA_COLOR)]:
    data = weekend_comparison[weekend_comparison['Origin'] == airport]
    fig.add_trace(
        go.Bar(
            x=data['DayType'],
            y=data['AvgDelay'],
            name=airport,
            marker_color=color,
            text=data['AvgDelay'].round(1),
            texttemplate='%{text} min',
            textposition='outside',
            hovertemplate='<b>%{x}</b><br>Avg Delay: %{y:.1f} min<extra></extra>'
        ),
        row=1, col=1
    )

# OTP
for airport, color in [('MCO', MCO_COLOR), ('MIA', MIA_COLOR)]:
    data = weekend_comparison[weekend_comparison['Origin'] == airport]
    fig.add_trace(
        go.Bar(
            x=data['DayType'],
            y=data['OTP'],
            name=airport,
            marker_color=color,
            text=data['OTP'].round(1),
            texttemplate='%{text}%',
            textposition='outside',
            showlegend=False,
            hovertemplate='<b>%{x}</b><br>OTP: %{y:.1f}%<extra></extra>'
        ),
        row=1, col=2
    )

fig.update_xaxes(title_text='Day Type', row=1, col=1)
fig.update_xaxes(title_text='Day Type', row=1, col=2)
fig.update_yaxes(title_text='Minutes', row=1, col=1)
fig.update_yaxes(title_text='Percentage (%)', row=1, col=2)

fig.update_layout(
    title_text='Weekend vs Weekday Performance Comparison',
    template=TEMPLATE,
    height=500,
    barmode='group'
)

fig.show()

### 📊 Key Insights:
- **Weekends have BETTER performance**: Lower delays and higher OTP
- **Weekday avg delay: 8-11 minutes** vs Weekend: 5-8 minutes
- **Business travel adds complexity**: Weekday traffic more delay-prone
- **MCO maintains advantage** on both weekends and weekdays

---
# 5. Summary Statistics

In [ ]:
# Overall summary
summary = departures.groupby('Origin').agg(
    TotalDepartures=('FlightDate', 'count'),
    AvgDelay=('ArrDelay', 'mean'),
    MedianDelay=('ArrDelay', 'median'),
    CancellationRate=('Cancelled', lambda x: (x.sum() / len(x)) * 100),
    OTP=('OnTime', lambda x: (x.sum() / len(x)) * 100),
    AvgDistance=('Distance', 'mean'),
    UniqueDestinations=('Dest', 'nunique'),
    UniqueAirlines=('UniqueCarrier', 'nunique')
).round(2)

print("\n" + "="*60)
print("SUMMARY STATISTICS: MCO vs MIA (2004-2008)")
print("="*60)
print(summary.to_string())
print("="*60)


SUMMARY STATISTICS: MCO vs MIA (2004-2008)
        TotalDepartures  AvgDelay  MedianDelay  CancellationRate    OTP  AvgDistance  UniqueDestinations  UniqueAirlines
Origin                                                                                                                  
MCO              521651      7.54         -1.0              1.04  78.88       902.24                  97              18
MIA              283825     11.13          0.0              1.62  74.03      1037.20                  52              13


---

**Key Findings:**
- MCO has significantly higher flight volume than MIA
- MCO has better on-time performance
- Both airports show seasonal patterns (tourism impact)
- Late aircraft is a major delay cause
- Peak hours are early morning and late afternoon